# MBO pipeline smoke runs

Run all cells using the project `.venv` kernel. Offline checks block socket connections and Databento client creation. Assertions stop execution on failure.

**SYNTHETIC — pipeline checks only.** Prices and order flow below are invented. These are ES-shaped fixtures, not historical ES observations.

The fixture generator lives only in this notebook. Delete its section when no longer needed; keep the labelled files under `data/synthetic/`. Production code never imports or calls it. The separate real-data section is disabled unless explicitly enabled.

In [1]:
import hashlib
import json
import os
import shutil
import socket
import tempfile
from contextlib import ExitStack
from datetime import date
from importlib.metadata import version
from pathlib import Path
from types import SimpleNamespace as NS
from unittest.mock import patch

import databento as db
import databento_dbn as dbn
import zstandard as zstd

from mbo_lab.data import (
    DataError,
    Request,
    build_book,
    discover,
    download,
    estimate,
    inspect_file,
    load_deltas,
    summarize_book,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
print({name: version(name) for name in ("databento", "nautilus_trader", "mbo-lab")})
from mbo_lab.budget import initialize

offline = ExitStack()
test_budget_dir = offline.enter_context(tempfile.TemporaryDirectory())
test_budget_path = Path(test_budget_dir) / "synthetic-budget.jsonl"
initialize(test_budget_path, "1.00", "2099-01-01")
offline.enter_context(patch.dict(os.environ, {"MBO_BUDGET_LEDGER": str(test_budget_path)}))
offline.enter_context(
    patch.object(socket.socket, "connect", side_effect=AssertionError("OFFLINE: network forbidden"))
)
offline.enter_context(
    patch.object(
        socket, "create_connection", side_effect=AssertionError("OFFLINE: network forbidden")
    )
)
offline.enter_context(
    patch.object(
        db, "Historical", side_effect=AssertionError("OFFLINE: Databento client forbidden")
    )
)
try:
    discover()
except AssertionError as exc:
    assert "Databento client forbidden" in str(exc)
else:
    raise AssertionError("offline discover unexpectedly created a client")
print("Offline guard active")

{'databento': '0.86.0', 'nautilus_trader': '1.231.0', 'mbo-lab': '0.1.0'}
Offline guard active


## Disposable synthetic fixture generator

This section writes SDK-encoded DBN metadata and records with Zstandard checksums. The final fill-attribution record deliberately carries F_LAST without producing a Nautilus delta. All fixture generation is removable; none belongs to the production package.

In [2]:
T = 1718582400000000000
SYMBOL = "SYNTH_ES"
SYNTHETIC = ROOT / "data" / "synthetic"
SYNTHETIC.mkdir(parents=True, exist_ok=True)
MBO = SYNTHETIC / "SYNTH_ES.mbo.dbn.zst"
DEFINITIONS = SYNTHETIC / "SYNTH_ES.definition.dbn.zst"
COMMON = dict(
    dataset="GLBX.MDP3",
    start=T,
    end=T + 60_000_000_000,
    stype_in=dbn.SType.RAW_SYMBOL,
    stype_out=dbn.SType.INSTRUMENT_ID,
    symbols=[SYMBOL],
)
MBO_META = dbn.Metadata(
    **COMMON,
    schema=dbn.Schema.MBO,
    mappings=[
        NS(
            raw_symbol=SYMBOL,
            intervals=[
                NS(start_date=date(2024, 6, 17), end_date=date(2024, 6, 18), symbol="12345")
            ],
        )
    ],
)
DEF_META = dbn.Metadata(**COMMON, schema=dbn.Schema.DEFINITION)


def mbo_record(
    action,
    order_id=0,
    price=None,
    size=0,
    side="N",
    flags=dbn.F_LAST,
    sequence=0,
    instrument_id=12345,
):
    return dbn.MBOMsg(
        publisher_id=1,
        instrument_id=instrument_id,
        ts_event=T + sequence * 1_000_000,
        ts_recv=T + sequence * 1_000_000,
        order_id=order_id,
        price=dbn.UNDEF_PRICE if price is None else int(price * 1_000_000_000),
        size=size,
        action=dbn.Action(action),
        side=dbn.Side(side),
        flags=flags,
        sequence=sequence,
        channel_id=0,
    )


S, L = dbn.F_SNAPSHOT, dbn.F_LAST
rows = [
    ("R", 0, None, 0, "N", S),
    ("A", 101, 6000, 5, "B", S),
    ("A", 102, 6000, 3, "B", S),
    ("A", 201, 6000.25, 4, "A", S),
    ("A", 202, 6000.50, 6, "A", S | L),
    ("A", 103, 5999.75, 2, "B", L),
    ("M", 101, 6000, 7, "B", L),
    ("C", 102, 6000, 3, "B", L),
    ("T", 0, 6000.25, 2, "B", 0),
    ("F", 201, 6000.25, 2, "A", 0),
    ("M", 201, 6000.25, 2, "A", L),
    ("F", 999, 5990, 2, "A", L),
    ("R", 0, None, 0, "N", S),
    ("A", 301, 6001, 4, "B", S),
    ("A", 302, 6001, 3, "B", S),
    ("A", 401, 6001.25, 8, "A", S | L),
    ("M", 301, 6001, 6, "B", L),
    ("C", 302, 6001, 3, "B", L),
    ("A", 303, 6000.75, 2, "B", L),
    ("F", 888, 5990, 1, "A", L),
]
records = [mbo_record(*row, sequence=i) for i, row in enumerate(rows)]
definition = dbn.InstrumentDefMsg(
    publisher_id=1,
    instrument_id=12345,
    ts_event=T,
    ts_recv=T,
    min_price_increment=250_000_000,
    display_factor=1_000_000_000,
    raw_symbol=SYMBOL,
    asset="ES",
    security_type="FUT",
    instrument_class=dbn.InstrumentClass.FUTURE,
    security_update_action=dbn.SecurityUpdateAction.ADD,
    expiration=T + 90 * 86400_000_000_000,
    activation=T - 90 * 86400_000_000_000,
    currency="USD",
    exchange="XCME",
    unit_of_measure_qty=50_000_000_000,
    min_lot_size_round_lot=1,
)


def write_dbn(path, metadata, messages):
    raw = metadata.encode() + b"".join(bytes(record) for record in messages)
    path.write_bytes(zstd.ZstdCompressor(write_checksum=True).compress(raw))
    return path


write_dbn(MBO, MBO_META, records)
write_dbn(DEFINITIONS, DEF_META, [definition])
manifest = {
    "synthetic": True,
    "purpose": "Pipeline checks only; invented ES-shaped order flow",
    "generator": "notebooks/smoke.ipynb — disposable fixture section",
    "instrument": "SYNTH_ES.GLBX",
    "sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (MBO, DEFINITIONS)},
}
(SYNTHETIC / "SYNTH_ES.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("SYNTHETIC — pipeline checks only")
print(manifest)

SYNTHETIC — pipeline checks only
{'synthetic': True, 'purpose': 'Pipeline checks only; invented ES-shaped order flow', 'generator': 'notebooks/smoke.ipynb — disposable fixture section', 'instrument': 'SYNTH_ES.GLBX', 'sha256': {'SYNTH_ES.mbo.dbn.zst': '0926a8ca6458daef24e41c4e79198c9d7bd21f15a5f264ed82396b730a0eb8db', 'SYNTH_ES.definition.dbn.zst': '3341e69dc120a7b97eabf162322059cd726a34f93df9b92465c65f395a8bc114'}}


## Real loader and exact book assertions

The pre-reset prefix verifies modifications, cancellation and fill attribution before the second snapshot can hide any errors. The full fixture verifies reset/recovery and deterministic reconstruction.

In [3]:
info = inspect_file(MBO)
instrument, deltas = load_deltas(MBO, DEFINITIONS)
assert str(instrument.id) == "SYNTH_ES.GLBX"
assert str(instrument.price_increment) == "0.25"
assert len(deltas) == 16
assert info["actions"] == {"R": 2, "A": 9, "M": 3, "C": 2, "T": 1, "F": 3}
assert not any(d.order.order_id in (888, 999) for d in deltas)
book = build_book(MBO, DEFINITIONS)
summary = summarize_book(book)
assert summary["bids"] == [
    {"price": "6001.00", "size": "6", "orders": 1},
    {"price": "6000.75", "size": "2", "orders": 1},
]
assert summary["asks"] == [{"price": "6001.25", "size": "8", "orders": 1}]
assert summary["spread"] == "0.25"
assert summary == summarize_book(build_book(MBO, DEFINITIONS))
with tempfile.TemporaryDirectory() as tmp:
    prefix = write_dbn(Path(tmp) / "prefix.dbn.zst", MBO_META, records[:12])
    before_reset = summarize_book(build_book(prefix, DEFINITIONS))
    assert before_reset["bids"] == [
        {"price": "6000.00", "size": "7", "orders": 1},
        {"price": "5999.75", "size": "2", "orders": 1},
    ]
    assert before_reset["asks"] == [
        {"price": "6000.25", "size": "2", "orders": 1},
        {"price": "6000.50", "size": "6", "orders": 1},
    ]
print(json.dumps(summary, indent=2))
print("PASS: real decoding, exact levels, fills, reset, deterministic reconstruction")

{
  "instrument": "SYNTH_ES.GLBX",
  "synthetic": true,
  "book_type": "L3_MBO",
  "delta_count": 16,
  "integrity": "passed",
  "best_bid": "6001.00",
  "best_ask": "6001.25",
  "spread": "0.25",
  "bids": [
    {
      "price": "6001.00",
      "size": "6",
      "orders": 1
    },
    {
      "price": "6000.75",
      "size": "2",
      "orders": 1
    }
  ],
  "asks": [
    {
      "price": "6001.25",
      "size": "8",
      "orders": 1
    }
  ]
}
PASS: real decoding, exact levels, fills, reset, deterministic reconstruction


## Explicit input failures

Malformed inputs must fail before a book is returned. Corrupt variants live in an automatically removed temporary directory.

In [4]:
def fails(fragment, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except DataError as exc:
        assert fragment.lower() in str(exc).lower(), str(exc)
        print(f"PASS: {exc}")
    else:
        raise AssertionError(f"Expected DataError containing {fragment!r}")


with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    missing = write_dbn(tmp / "missing.dbn.zst", MBO_META, records[1:])
    fails("initialisation", build_book, missing, DEFINITIONS)
    incomplete = write_dbn(tmp / "incomplete.dbn.zst", MBO_META, records[:10])
    fails("ending event", build_book, incomplete, DEFINITIONS)
    wrong = write_dbn(
        tmp / "wrong.dbn.zst",
        MBO_META,
        [
            mbo_record("R", flags=S | L, instrument_id=99999),
        ],
    )
    fails("mismatched", build_book, wrong, DEFINITIONS)
    truncated = tmp / "truncated.dbn.zst"
    truncated.write_bytes(MBO.read_bytes()[:-1])
    fails("truncated", build_book, truncated, DEFINITIONS)
    short_record = tmp / "record.dbn.zst"
    short_record.write_bytes(
        zstd.ZstdCompressor().compress(MBO_META.encode() + bytes(records[0])[:-1])
    )
    fails("truncated", build_book, short_record, DEFINITIONS)
    empty = write_dbn(tmp / "empty.dbn.zst", MBO_META, [])
    fails("no records", build_book, empty, DEFINITIONS)
    # A complete raw F_LAST on a non-delta record must still close the event.
    ending_fill = write_dbn(
        tmp / "ending-fill.dbn.zst",
        MBO_META,
        [
            records[0],
            records[1],
            records[2],
            mbo_record("F", 999, 5990, 1, "A", L, sequence=3),
        ],
    )
    assert summarize_book(build_book(ending_fill, DEFINITIONS))["bids"][0]["size"] == "8"
print("PASS: malformed input rejection and raw non-delta boundary")

PASS: Missing initialisation: first MBO record must be snapshot CLEAR
PASS: Incomplete ending event: final raw MBO record lacks F_LAST
PASS: Mismatched instruments/publishers; supply exactly one contract
PASS: Truncated Zstandard frame: C:\Users\ADARSH~1\AppData\Local\Temp\tmp1gfhfh6z\truncated.dbn.zst
PASS: Truncated DBN metadata or record: C:\Users\ADARSH~1\AppData\Local\Temp\tmp1gfhfh6z\record.dbn.zst
PASS: DBN file has no metadata or no records
PASS: malformed input rejection and raw non-delta boundary


## Discovery, budget and cache checks without network

A fake transport copies the labelled fixture files; the real DBN validation and book loader still run. A second download is required to work with Databento client construction forbidden.

In [5]:
request = Request("SYNTH_ES", "2024-06-17T00:00:00Z", "2024-06-17T00:01:00Z")
calls = []


def fake_range(**kwargs):
    calls.append(kwargs["schema"])
    source = DEFINITIONS if kwargs["schema"] == "definition" else MBO
    assert kwargs["symbols"] in [["SYNTH_ES"], ["12345"]]
    if kwargs["stype_in"] == "instrument_id":
        assert kwargs["symbols"] == ["12345"]
        metadata = dbn.Metadata(
            **{**COMMON, "stype_in": dbn.SType.INSTRUMENT_ID, "symbols": ["12345"]},
            schema=dbn.Schema(kwargs["schema"]),
            mappings=[
                NS(
                    raw_symbol="12345",
                    intervals=[
                        NS(start_date=date(2024, 6, 17), end_date=date(2024, 6, 18), symbol="12345")
                    ],
                )
            ],
        )
        write_dbn(
            kwargs["path"], metadata, [definition] if kwargs["schema"] == "definition" else records
        )
    else:
        shutil.copyfile(source, kwargs["path"])


client = NS(
    metadata=NS(
        list_datasets=lambda: ["GLBX.MDP3"],
        list_schemas=lambda **kw: ["mbo", "definition"],
        get_dataset_range=lambda **kw: {"start": "2024-01-01", "end": "2024-12-31"},
        list_publishers=lambda: [{"publisher_id": 1}],
        get_cost=lambda **kw: 0.01,
    ),
    timeseries=NS(get_range=fake_range),
    symbology=NS(
        resolve=lambda **kw: {
            "result": {
                "ES.c.0": [{"d0": "2024-06-17", "d1": "2024-06-18", "s": "12345"}],
            }
        }
    ),
)
assert "mbo" in discover(client=client)["schemas"]
assert estimate(request, client=client)["total_usd"] == 0.02
continuous = Request("ES.c.0", request.start, request.end, stype_in="continuous")
assert estimate(continuous, client=client)["request"] == {
    "symbol": "12345",
    "start": request.start,
    "end": request.end,
    "dataset": request.dataset,
    "stype_in": "instrument_id",
}
with tempfile.TemporaryDirectory() as tmp:
    resolved = download(continuous, tmp, max_cost=0.02, client=client, confirm=lambda details: True)
    assert not resolved["cached"]
    assert resolved["manifest"]["synthetic"]
    assert resolved["manifest"]["resolved_request"]["symbol"] == "12345"
    assert (
        summarize_book(build_book(resolved["paths"]["mbo"], resolved["paths"]["definition"]))
        == summary
    )
calls.clear()
with tempfile.TemporaryDirectory() as tmp:
    fails("exceeds", download, request, tmp, max_cost=0.001, client=client)
    assert calls == []
    first = download(request, tmp, max_cost=0.02, client=client, confirm=lambda details: True)
    assert calls == ["definition", "mbo"] and not first["cached"]
    cached = download(request, tmp, max_cost=0)
    assert cached["cached"] and calls == ["definition", "mbo"]
    assert (
        summarize_book(build_book(cached["paths"]["mbo"], cached["paths"]["definition"])) == summary
    )
    cached["paths"]["mbo"].write_bytes(b"broken")
    fails("changed", download, request, tmp, max_cost=1, client=client, confirm=lambda details: True)
    assert calls == ["definition", "mbo"]
print("PASS: discovery, estimates, cost cap, offline cache, corrupt-cache rejection")
numeric = Request("12345", request.start, request.end, stype_in="instrument_id")
with patch.object(client.symbology, "resolve", side_effect=AssertionError("already resolved")):
    with tempfile.TemporaryDirectory() as tmp:
        result = download(numeric, tmp, max_cost=0.02, client=client, confirm=lambda details: True)
        assert (
            summarize_book(build_book(result["paths"]["mbo"], result["paths"]["definition"]))
            == summary
        )
        assert result["manifest"]["synthetic"]
print("PASS: numeric DBN metadata, definition identity/precision, synthetic provenance")

PASS: Estimated $0.020000 exceeds maximum $0.001000
PASS: Cached files are missing or changed; inspect them before redownloading
PASS: discovery, estimates, cost cap, offline cache, corrupt-cache rejection
PASS: numeric DBN metadata, definition identity/precision, synthetic provenance


## CLI and storage compatibility

Run the CLI entrypoint under the same offline guard; verify plain DBN and multiple Zstandard frames as well as checksum rejection.

In [6]:
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO

from mbo_lab.cli import main

output = StringIO()
with redirect_stdout(output):
    assert main(["book", "--mbo", str(MBO), "--definitions", str(DEFINITIONS)]) == 0
assert json.loads(output.getvalue()) == summary
output = StringIO()
with redirect_stdout(output):
    assert main(["inspect", "--file", str(MBO)]) == 0
assert json.loads(output.getvalue())["records"] == 20
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    uncompressed = tmp / "plain.dbn"
    raw = MBO_META.encode() + b"".join(bytes(r) for r in records)
    uncompressed.write_bytes(raw)
    assert inspect_file(uncompressed)["records"] == 20
    multiframe = tmp / "multiframe.dbn.zst"
    cut = len(raw) // 2
    compressor = zstd.ZstdCompressor(write_checksum=True)
    multiframe.write_bytes(compressor.compress(raw[:cut]) + compressor.compress(raw[cut:]))
    assert inspect_file(multiframe)["records"] == 20
    error = StringIO()
    with redirect_stderr(error):
        assert (
            main(["book", "--mbo", str(tmp / "absent.dbn"), "--definitions", str(DEFINITIONS)]) == 1
        )
    assert "Cannot read complete DBN" in error.getvalue()
    changed = bytearray(MBO.read_bytes())
    changed[-1] ^= 1
    corrupt = tmp / "checksum.dbn.zst"
    corrupt.write_bytes(changed)
    fails("Cannot read", inspect_file, corrupt)
fails("weekday", Request, "ESM4", "2024-06-17T01:00:00Z", "2024-06-17T01:01:00Z")
fails("one GLBX", Request, "ALL_SYMBOLS", request.start, request.end)
print(
    "PASS: CLI output/errors, uncompressed and multi-frame inspection, checksum, request validation"
)
offline.close()
print("Offline checks complete; network guard removed")

PASS: Cannot read complete DBN file C:\Users\ADARSH~1\AppData\Local\Temp\tmpjv8_qqan\checksum.dbn.zst: zstd decompressor error: Restored data doesn't match checksum
PASS: Start at weekday 00:00:00Z to include the CME MBO snapshot
PASS: This initial pipeline requires one GLBX.MDP3 futures symbol
PASS: CLI output/errors, uncompressed and multi-frame inspection, checksum, request validation
Offline checks complete; network guard removed


## Optional real ES smoke run

Disabled by default. Set `MBO_RUN_LIVE=1`, `DATABENTO_API_KEY`, and `MBO_MAX_COST_USD` before launching the notebook kernel. The cost cap applies to the estimated combined MBO and definition request. Historical downloads can incur charges. A verified cache hit does not download again.

The example uses a one-minute interval including the weekday midnight snapshot. Review `es.toml` first. No synthetic fallback is permitted.

In [7]:
from mbo_lab.cli import confirm_download

if os.environ.get("MBO_RUN_LIVE") != "1":
    print("SKIPPED: real ES smoke run (set MBO_RUN_LIVE=1 explicitly)")
else:
    assert os.environ.get("DATABENTO_API_KEY"), "Configure DATABENTO_API_KEY locally"
    assert os.environ.get("MBO_MAX_COST_USD"), "Specify your authorised USD cap"
    live_request = Request.from_toml(ROOT / "es.toml")
    live_discovery = discover()
    assert live_request.dataset in live_discovery["datasets"]
    assert "mbo" in live_discovery["schemas"]
    live_quote = estimate(live_request)
    print("Estimate:", live_quote)
    result = download(
        live_request, ROOT / "data" / "raw", max_cost=float(os.environ["MBO_MAX_COST_USD"]), confirm=confirm_download
    )
    live_book = build_book(result["paths"]["mbo"], result["paths"]["definition"])
    live_summary = summarize_book(live_book)
    assert not live_summary["synthetic"]
    assert live_summary["bids"] and live_summary["asks"], "Expected a populated ES book"
    print(json.dumps(live_summary, indent=2))
    print("PASS: REAL ES discovery -> estimate -> download -> populated L3 book")

SKIPPED: real ES smoke run (set MBO_RUN_LIVE=1 explicitly)
